In [11]:
import pandas as pd
from neo4j import GraphDatabase

In [22]:
# --- Neo4j Connection Details ---
#URI = "bolt://localhost:7687" # Default for Neo4j Desktop
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "12345678")
driver = GraphDatabase.driver(URI, auth=AUTH) 

In [13]:
def run_cypher_query(query, params=None):
    """Function to run a single Cypher query and return the results."""
    with driver.session() as session:
        result = session.run(query, params)
        return [record for record in result]
print("Neo4j Driver initialized.")

Neo4j Driver initialized.


## Define and Create Constraints and Indexes (Cypher)

In [15]:
# # Cypher Queries for Constraints and Indexes
# schema_queries = [
#     # 1. Visitor Node Constraint (Unique visitorId)
#     "CREATE CONSTRAINT visitorId_is_unique IF NOT EXISTS ON (v:Visitor) ASSERT v.visitorId IS UNIQUE;",

#     # 2. Session Node Constraint (Unique sid)
#     "CREATE CONSTRAINT sid_is_unique IF NOT EXISTS ON (s:Session) ASSERT s.sid IS UNIQUE;",

#     # 3. Product Node Constraint (Unique pid)
#     "CREATE CONSTRAINT pid_is_unique IF NOT EXISTS ON (p:Product) ASSERT p.pid IS UNIQUE;",

#     # 4. Category Node Constraint (Unique catid)
#     "CREATE CONSTRAINT catid_is_unique IF NOT EXISTS ON (c:Category) ASSERT c.catid IS UNIQUE;",

#     # 5. PropertyValue Node Constraint (Unique value)
#     "CREATE CONSTRAINT prop_val_is_unique IF NOT EXISTS ON (pv:PropertyValue) ASSERT pv.value IS UNIQUE;",

#     # 6. PropertyType Index (To quickly look up property names)
#     "CREATE INDEX prop_type_index IF NOT EXISTS FOR (pt:PropertyType) ON (pt.name);"
# ]

# # Execute the queries
# print("Creating Neo4j Constraints and Indexes...")
# for query in schema_queries:
#     try:
#         run_cypher_query(query)
#         print(f"  - SUCCESS: {query.split(' ')[2]}")
#     except Exception as e:
#         print(f"  - ERROR with query: {query.split(' ')[2]} - {e}")

# print("Neo4j schema foundation is complete.")



# Cypher Queries for Constraints and Indexes (UPDATED FOR NEO4J 5+)
schema_queries = [
    # 1. Visitor Node Constraint (Unique visitorId)
    "CREATE CONSTRAINT visitorId_is_unique IF NOT EXISTS FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE;",

    # 2. Session Node Constraint (Unique sid)
    "CREATE CONSTRAINT sid_is_unique IF NOT EXISTS FOR (s:Session) REQUIRE s.sid IS UNIQUE;",

    # 3. Product Node Constraint (Unique pid)
    "CREATE CONSTRAINT pid_is_unique IF NOT EXISTS FOR (p:Product) REQUIRE p.pid IS UNIQUE;",

    # 4. Category Node Constraint (Unique catid)
    "CREATE CONSTRAINT catid_is_unique IF NOT EXISTS FOR (c:Category) REQUIRE c.catid IS UNIQUE;",

    # 5. PropertyValue Node Constraint (Unique value)
    "CREATE CONSTRAINT prop_val_is_unique IF NOT EXISTS FOR (pv:PropertyValue) REQUIRE pv.value IS UNIQUE;",

    # 6. PropertyType Index (This one was fine, but we keep it)
    "CREATE INDEX prop_type_index IF NOT EXISTS FOR (pt:PropertyType) ON (pt.name);"
]

# Execute the queries
print("Creating Neo4j Constraints and Indexes...")
for query in schema_queries:
    try:
        run_cypher_query(query)
        # Use split and list comprehension to handle the output gracefully
        parts = query.split(' ')
        label_prop = next(p for p in parts if ')' in p)
        print(f"  - SUCCESS: {parts[2]} on {label_prop}")
    except Exception as e:
        print(f"  - ERROR with query: {query.split(' ')[2]} - {e}")

print("Neo4j schema foundation is complete.")

Creating Neo4j Constraints and Indexes...
  - SUCCESS: visitorId_is_unique on (v:Visitor)
  - SUCCESS: sid_is_unique on (s:Session)
  - SUCCESS: pid_is_unique on (p:Product)
  - SUCCESS: catid_is_unique on (c:Category)
  - SUCCESS: prop_val_is_unique on (pv:PropertyValue)
  - SUCCESS: prop_type_index on (pt:PropertyType)
Neo4j schema foundation is complete.


## 2.1 Load Data and Preliminary Cleaning

In [12]:
# Load the datasets
events_df = pd.read_csv('data/events.csv')
item_properties1_df = pd.read_csv('data/item_properties_part1.csv')
item_properties2_df = pd.read_csv('data/item_properties_part2.csv')
category_tree_df = pd.read_csv('data/category_tree.csv')

# Combine item properties (part 1 and 2)
item_properties_df = pd.concat([item_properties1_df, item_properties2_df], ignore_index=True)

# Rename columns for clarity and consistency with the graph model
events_df = events_df.rename(columns={'timestamp': 'timestamp_ms', 'visitorid': 'visitorId', 'itemid': 'pid', 'event': 'action'})
category_tree_df = category_tree_df.rename(columns={'categoryid': 'catid', 'parentid': 'parentCatid'})

# Convert timestamps from milliseconds to seconds
# Neo4j handles POSIX timestamps in milliseconds, but conversion is useful for internal duration calc.
events_df['timestamp_s'] = (events_df['timestamp_ms'] / 1000).astype(int)
item_properties_df['timestamp_s'] = (item_properties_df['timestamp'] / 1000).astype(int)

print(f"Events loaded: {len(events_df)} rows")
print(f"Item Properties loaded: {len(item_properties_df)} rows")
print(f"Category Tree loaded: {len(category_tree_df)} rows")

Events loaded: 2756101 rows
Item Properties loaded: 20275902 rows
Category Tree loaded: 1669 rows


## 2.2 Sessionization Logic

In [13]:
SESSION_TIMEOUT_S = 5 * 60  # 5 minutes in seconds

# Sort events by visitor and then by time
events_df = events_df.sort_values(['visitorId', 'timestamp_s']).reset_index(drop=True)

# Calculate the time difference (delta) between consecutive events for each visitor
events_df['time_diff'] = events_df.groupby('visitorId')['timestamp_s'].diff().fillna(0)

# Identify the start of a new session: either the first event for a visitor (time_diff=0)
# or if the time difference is greater than the session timeout.
events_df['new_session'] = events_df['time_diff'] > SESSION_TIMEOUT_S

# Create a session counter that increments every time 'new_session' is True
events_df['session_count'] = events_df.groupby('visitorId')['new_session'].cumsum()

# Create the unique Session ID (sid)
events_df['sid'] = events_df['visitorId'].astype(str) + '_' + events_df['session_count'].astype(str)

# Calculate the sequential step for the PERFOMED relationship
events_df['step'] = events_df.groupby('sid').cumcount() + 1

# Extract the unique Session starting/ending times for the Session nodes
session_nodes_df = events_df.groupby('sid').agg(
    visitorId=('visitorId', 'first'),
    startTime=('timestamp_ms', 'min'),
    endTime=('timestamp_ms', 'max'),
).reset_index()

print(f"Total Sessions created: {len(session_nodes_df)}")
events_df[['visitorId', 'sid', 'timestamp_s', 'time_diff', 'new_session', 'step', 'action', 'pid']].head(10)

Total Sessions created: 1930778


,visitorId,sid,timestamp_s,time_diff,new_session,step,action,pid
0,0,0_0,1442004589,0.0,False,1,view,285930
1,0,0_0,1442004759,170.0,False,2,view,357564
2,0,0_0,1442004917,158.0,False,3,view,67045
3,1,1_0,1439487966,0.0,False,1,view,72028
4,2,2_0,1438969904,0.0,False,1,view,325215
5,2,2_0,1438970013,109.0,False,2,view,325215
6,2,2_0,1438970212,199.0,False,3,view,259884
7,2,2_0,1438970468,256.0,False,4,view,216305
8,2,2_1,1438970905,437.0,True,1,view,342816
9,2,2_2,1438971444,539.0,True,1,view,342816


## Part 3: Optimized Graph Loading 

In [19]:
from tqdm import tqdm

In [20]:
def load_data_to_neo4j(df, cypher_query, batch_size=10000, key='data'):
    """Utility to batch and load data from a DataFrame to Neo4j."""
    total_batches = len(df) // batch_size + (1 if len(df) % batch_size else 0)

    with tqdm(total=total_batches, desc=f"Loading {key}") as pbar:
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i + batch_size].to_dict('records')
            run_cypher_query(cypher_query, {key: batch})
            pbar.update(1)

In [23]:
# -----------------------------------------------------------
# 1. Load Visitor and Session Nodes & HAS_SESSION
# -----------------------------------------------------------
session_cypher = """
UNWIND $data AS row
MERGE (v:Visitor {visitorId: row.visitorId})
MERGE (s:Session {sid: row.sid})
SET s.startTime = row.startTime,
    s.endTime = row.endTime
MERGE (v)-[:HAS_SESSION]->(s)
"""
# Use session_nodes_df from Part 2.2
load_data_to_neo4j(session_nodes_df, session_cypher, batch_size=10000, key='data')
print("Loaded Visitor and Session Nodes.")

# -----------------------------------------------------------
# 2. Load Product Nodes
# -----------------------------------------------------------
unique_pids = events_df[['pid']].drop_duplicates()
product_cypher = """
UNWIND $data AS row
MERGE (p:Product {pid: row.pid})
"""
load_data_to_neo4j(unique_pids, product_cypher, batch_size=10000, key='data')
print("Loaded Product Nodes.")

# -----------------------------------------------------------
# 3. Load PERFORMED Relationships (The sequential session actions)
# -----------------------------------------------------------
events_load_df = events_df[['sid', 'pid', 'step', 'action', 'timestamp_ms']]

performed_cypher = """
UNWIND $data AS row
MATCH (s:Session {sid: row.sid})
MATCH (p:Product {pid: row.pid})
CREATE (s)-[:PERFORMED {
    step: row.step, 
    action: row.action, 
    timestamp: row.timestamp_ms
}]->(p)
"""
# THIS WILL BE THE LONGEST STEP
load_data_to_neo4j(events_load_df, performed_cypher, batch_size=10000, key='data')
print("Loaded PERFORMED relationships.")

# -----------------------------------------------------------
# 4. Load Category Hierarchy and Links
# -----------------------------------------------------------
# IS_PARENT_OF
category_links = category_tree_df.dropna(subset=['parentCatid']).to_dict('records')
category_hierarchy_cypher = """
UNWIND $data AS row
MERGE (child:Category {catid: row.catid})
MERGE (parent:Category {catid: row.parentCatid})
MERGE (parent)-[:IS_PARENT_OF]->(child)
"""
# This file is small, running directly without advanced batching
run_cypher_query(category_hierarchy_cypher, {'data': category_links}) 
print("Loaded Category Nodes and IS_PARENT_OF relationships.")

# BELONGS_TO (Product to Category)
item_cat_df = item_properties_df[item_properties_df['property'] == 'categoryid'].copy()
item_cat_df = item_cat_df.sort_values('timestamp').drop_duplicates(subset=['itemid'], keep='last')
item_cat_df = item_cat_df.rename(columns={'itemid': 'pid', 'value': 'catid_str'})
item_cat_df['catid'] = item_cat_df['catid_str'].str.replace('n', '').str.split('.').str[0].astype(int)

belongs_to_cypher = """
UNWIND $data AS row
MATCH (p:Product {pid: row.pid})
MATCH (c:Category {catid: row.catid})
MERGE (p)-[:BELONGS_TO]->(c)
"""
load_data_to_neo4j(item_cat_df[['pid', 'catid']], belongs_to_cypher, batch_size=10000, key='data')
print("Loaded BELONGS_TO relationships.")


# -----------------------------------------------------------
# 5. Load Item Properties (Context Layer)
# -----------------------------------------------------------
prop_load_df = item_properties_df.rename(columns={'itemid': 'pid', 'property': 'prop_name', 'value': 'prop_value', 'timestamp': 'timestamp_ms'})

properties_cypher = """
UNWIND $data AS row
MATCH (p:Product {pid: row.pid})

// Create or Match the PropertyType node
MERGE (pt:PropertyType {name: row.prop_name})

// Create or Match the PropertyValue node
MERGE (pv:PropertyValue {value: row.prop_value})

// Link PropertyValue to PropertyType
MERGE (pv)-[:IS_TYPE_OF]->(pt) 

// Link Product to PropertyValue with a timestamp property
CREATE (p)-[:HAS_PROPERTY {
    timestamp: row.timestamp_ms
}]->(pv)
"""
load_data_to_neo4j(prop_load_df[['pid', 'prop_name', 'prop_value', 'timestamp_ms']], properties_cypher, batch_size=10000, key='data')
print("Loaded Item Properties and HAS_PROPERTY relationships.")

print("\n--- CORE GRAPH MODELING COMPLETE ---")

Loading data: 100%|██████████| 194/194 [05:35<00:00,  1.73s/it]


Loaded Visitor and Session Nodes.


Loading data: 100%|██████████| 24/24 [00:19<00:00,  1.25it/s]


Loaded Product Nodes.


Loading data: 100%|██████████| 276/276 [09:41<00:00,  2.11s/it]


Loaded PERFORMED relationships.
Loaded Category Nodes and IS_PARENT_OF relationships.


Loading data: 100%|██████████| 42/42 [01:09<00:00,  1.65s/it]


Loaded BELONGS_TO relationships.


Loading data: 100%|██████████| 2028/2028 [57:27<00:00,  1.70s/it] 


Loaded Item Properties and HAS_PROPERTY relationships.

--- CORE GRAPH MODELING COMPLETE ---


## Create the CO_OCCURS_WITH Relationship

In [30]:
import pandas as pd
from neo4j import GraphDatabase, exceptions

# --- Neo4j Connection Details: REVERTING TO DIRECT BOLT PROTOCOL ---
URI = "bolt://127.0.0.1:7687"  # Changed from neo4j:// to bolt://
AUTH = ("neo4j", "12345678") # USE YOUR ACTUAL PASSWORD

# Initialize the driver with a high connection timeout
# We will use the default driver settings here, relying on the session timeout for the long query
driver = GraphDatabase.driver(URI, auth=AUTH)

def run_cypher_query(query, params=None, timeout_s=3600): # Set the default query timeout to 1 hour
    """Function to run a single Cypher query and return the results."""
    with driver.session() as session:
        # We explicitly set the query timeout on the session.run call
        result = session.run(query, params, timeout=timeout_s)
        return [record for record in result]

print("Neo4j Driver initialized using BOLT protocol with 1 hour query timeout.")

Neo4j Driver initialized using BOLT protocol with 1 hour query timeout.


In [4]:
import pandas as pd
from neo4j import GraphDatabase, exceptions

# --- Part 4: Create the CO_OCCURS_WITH Relationship (The Recommendation Core) ---

# Re-initialize the driver connection since we closed it in the previous step's code block
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "12345678") # USE YOUR ACTUAL PASSWORD
driver = GraphDatabase.driver(URI, auth=AUTH)

def run_cypher_query(query, params=None):
    """Function to run a single Cypher query and return the results."""
    with driver.session() as session:
        result = session.run(query, params)
        return [record for record in result]

co_occurrence_cypher = """
// Set a timeout for the query. This is a large aggregation and might take minutes.
// CALL apoc.cypher.runTimeboxed('
MATCH (s:Session)-[r1:PERFORMED]->(p1:Product)
MATCH (s)-[r2:PERFORMED]->(p2:Product)
// Ensure the two products are NOT the same (pure co-occurrence pair)
WHERE id(p1) < id(p2)

// Aggregate the session counts and collected durations for the pair
WITH p1, p2, 
     COUNT(s) AS popularity, 
     // Collect the absolute time difference between the two events in the session (for Bonus)
     COLLECT(toInteger(abs(r1.timestamp - r2.timestamp))) AS durations
     
// Calculate the average duration
WITH p1, p2, popularity, 
     reduce(s = 0, d IN durations | s + d) / size(durations) AS avgDurationMs

// Create the derived CO_OCCURS_WITH relationship
MERGE (p1)-[co:CO_OCCURS_WITH]->(p2)
SET co.popularity = popularity,
    co.avgDuration = avgDurationMs
// ', 300000) YIELD value 
""" # Using a comment for a possible timeout/APOC call if needed

print("\n--- Starting CO_OCCURS_WITH aggregation (This will be CPU intensive) ---")
try:
    # Run the single, large aggregation query
    # NOTE: This query will take several minutes to run on a large dataset.
    run_cypher_query(co_occurrence_cypher)
    print("SUCCESS: CO_OCCURS_WITH relationships created!")
    
except Exception as e:
    print(f"ERROR creating CO_OCCURS_WITH: {e}")

# Close the driver connection after the final load
driver.close()
print("\nDriver connection closed. Graph Modeling is fully complete.")


--- Starting CO_OCCURS_WITH aggregation (This will be CPU intensive) ---


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=7, column=7, offset=282>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 282, 'line': 7, 'column': 7}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// Set a timeout for the query. This is a large aggregation and might take minutes.\n// CALL apoc.cypher.runTimeboxed('\nMATCH (s:Session)-[r1:PERFORMED]->(p1:Product)\nMATCH (s)-[r2:PERFORMED]->(p2:Product)\n// Ensure the two products are NOT the same (pure co-occurrence pair)\nWHERE id(p1) < id(p2)\n\n// Aggr

SUCCESS: CO_OCCURS_WITH relationships created!

Driver connection closed. Graph Modeling is fully complete.


In [1]:
import pandas as pd
from neo4j import GraphDatabase, exceptions

# --- Part 4: Create the CO_OCCURS_WITH Relationship (The Recommendation Core) ---

# Re-initialize the driver connection
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "12345678")  # <-- Use your actual password
driver = GraphDatabase.driver(URI, auth=AUTH)

def run_cypher_query(query, params=None):
    """Function to run a single Cypher query and return the results."""
    with driver.session() as session:
        result = session.run(query, params)
        return [record for record in result]

# --- Updated Cypher query using elementId() instead of id() ---
co_occurrence_cypher = """
// CO_OCCURS_WITH aggregation between products based on shared sessions
MATCH (s:Session)-[r1:PERFORMED]->(p1:Product)
MATCH (s)-[r2:PERFORMED]->(p2:Product)

// Ensure the two products are NOT the same (pure co-occurrence pair)
WHERE elementId(p1) < elementId(p2)

// Aggregate the session counts and collected durations for the pair
WITH p1, p2, 
     COUNT(s) AS popularity, 
     COLLECT(toInteger(abs(r1.timestamp - r2.timestamp))) AS durations
     
// Calculate the average duration
WITH p1, p2, popularity, 
     CASE 
         WHEN size(durations) > 0 
         THEN reduce(total = 0, d IN durations | total + d) / size(durations)
         ELSE 0 
     END AS avgDurationMs

// Create or update the CO_OCCURS_WITH relationship
MERGE (p1)-[co:CO_OCCURS_WITH]->(p2)
SET co.popularity = popularity,
    co.avgDuration = avgDurationMs
"""

print("\n--- Starting CO_OCCURS_WITH aggregation (This will be CPU intensive) ---")
try:
    run_cypher_query(co_occurrence_cypher)
    print("✅ SUCCESS: CO_OCCURS_WITH relationships created!")

    # Optional: count created relationships
    count_result = run_cypher_query("MATCH ()-[r:CO_OCCURS_WITH]->() RETURN COUNT(r) AS count")
    print(f"🔗 Total CO_OCCURS_WITH relationships: {count_result[0]['count']}")

except Exception as e:
    print(f"❌ ERROR creating CO_OCCURS_WITH: {e}")

# Close the driver connection after the final load
driver.close()
print("\nDriver connection closed. Graph Modeling is fully complete.")



--- Starting CO_OCCURS_WITH aggregation (This will be CPU intensive) ---
✅ SUCCESS: CO_OCCURS_WITH relationships created!
🔗 Total CO_OCCURS_WITH relationships: 1153137

Driver connection closed. Graph Modeling is fully complete.


## Probable Next Event Prediction

In [2]:
from neo4j import GraphDatabase, exceptions

# --- CONNECTION PARAMETERS (Ensure this matches your setup) ---
URI = "bolt://127.0.0.1:7687"
AUTH = ("neo4j", "12345678")

# --- 5.1 (Revised): Next Event Prediction Query (Core Logic) ---

TARGET_PID = 119736

print(f"\n--- 5.1: Next Event Prediction (Test with pid = {TARGET_PID}) ---")

prediction_query = f"""
// Use the most popular product (119736) as the starting point
MATCH (p1:Product {{pid: {TARGET_PID}}})-[co:CO_OCCURS_WITH]-(p2:Product)
RETURN p2.pid AS recommended_pid, 
       co.popularity AS co_occurrence_count,
       co.avgDuration AS average_time_diff_ms
ORDER BY co_occurrence_count DESC
LIMIT 5
"""

# Re-initialize the driver DIRECTLY now that we know it was closed.
try:
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    # Define run_cypher_query again as it may have been defined outside the current scope
    def run_cypher_query(query, params=None, timeout_s=3600):
        with driver.session() as session:
            result = session.run(query, params, timeout=timeout_s)
            return [record for record in result]
            
    results = run_cypher_query(prediction_query)
    
    print(f"Top 5 Next-Item Recommendations for Product {TARGET_PID}:")
    
    # Convert milliseconds to a more readable format (seconds/minutes)
    for i, rec in enumerate(results):
        ms = rec['average_time_diff_ms']
        seconds = ms / 1000
        minutes = seconds / 60
        
        print(f"  {i+1}. PID: {rec['recommended_pid']:<7} | Popularity: {rec['co_occurrence_count']:<8} | Avg Time Diff: {minutes:.2f} min ({seconds:.0f}s)")
        
except Exception as e:
    print(f"ERROR running prediction query: {e}")

finally:
    if 'driver' in locals() and driver:
        driver.close()


--- 5.1: Next Event Prediction (Test with pid = 119736) ---
Top 5 Next-Item Recommendations for Product 119736:
  1. PID: 186702  | Popularity: 94       | Avg Time Diff: 2.62 min (157s)
  2. PID: 417927  | Popularity: 47       | Avg Time Diff: 2.38 min (143s)
  3. PID: 190000  | Popularity: 37       | Avg Time Diff: 3.62 min (217s)
  4. PID: 235559  | Popularity: 33       | Avg Time Diff: 1.48 min (89s)
  5. PID: 274443  | Popularity: 32       | Avg Time Diff: 0.75 min (45s)


## Recurring Event Patterns Query

In [6]:
# --- 5.2 (FINAL REVISED): Detecting Top 3 Recurring 3-Step Patterns (Any Sequence) ---

print("\n--- 5.2: Detecting Top 3 Recurring 3-Step Patterns (Final Corrected) ---")

pattern_query_final = """
// 1. Match the Session and the first Product (P1)
MATCH (s:Session)-[r1:PERFORMED]->(p1:Product)

// 2. Match the second Product (P2) using a relationship with the next step
MATCH (s)-[r2:PERFORMED]->(p2:Product)
WHERE r2.step = r1.step + 1

// 3. Match the third Product (P3) using a relationship with the subsequent step
MATCH (s)-[r3:PERFORMED]->(p3:Product)
WHERE r3.step = r2.step + 1

// Group and Count the sequences
WITH p1.pid AS pid_A, p2.pid AS pid_B, p3.pid AS pid_C, COUNT(s) AS pattern_count
ORDER BY pattern_count DESC
LIMIT 3
RETURN pid_A, pid_B, pid_C, pattern_count
"""

# Re-initialize driver
try:
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    # Run the query (This is highly efficient and should return results)
    pattern_results = run_cypher_query(pattern_query_final)
    
    print("Top 3 (Product A -> Product B -> Product C) Sequences:")
    
    for i, res in enumerate(pattern_results):
        print(f"  {i+1}. Pattern: {res['pid_A']} -> {res['pid_B']} -> {res['pid_C']} | Count: {res['pattern_count']}")
        
except Exception as e:
    print(f"ERROR running pattern query: {e}")

finally:
    if 'driver' in locals() and driver:
        driver.close()


--- 5.2: Detecting Top 3 Recurring 3-Step Patterns (Final Corrected) ---
Top 3 (Product A -> Product B -> Product C) Sequences:
  1. Pattern: 40870 -> 40870 -> 40870 | Count: 224
  2. Pattern: 461686 -> 461686 -> 461686 | Count: 148
  3. Pattern: 414432 -> 414432 -> 414432 | Count: 98


In [7]:
from neo4j import GraphDatabase, exceptions

# --- CONNECTION PARAMETERS (Ensure this matches your setup) ---
URI = "bolt://127.0.0.1:7687"
AUTH = ("neo4j", "12345678")
# --- 6. Hybrid Model: LLM Prompt Generation ---

print("\n--- 6. Generating LLM Prompt for Recommendation Explanation ---")

# We will use PID 119736 -> 186702 as the base
hybrid_query = """
MATCH (p_rec:Product {pid: 186702})-[:BELONGS_TO]->(c:Category)
WITH p_rec, c

// Get the properties for the viewed product (p_view)
MATCH (p_view:Product {pid: 119736})-[hp:HAS_PROPERTY]->(pv:PropertyValue)-[:IS_TYPE_OF]->(pt:PropertyType)

// Filter p_view properties to get only the most recent (highest timestamp)
WITH p_rec, c, p_view, pv, pt, hp
ORDER BY hp.timestamp DESC
LIMIT 5 

// Construct the prompt string using the COLLECT function
WITH 'Recommended Product (' + toString(p_rec.pid) + ') in Category ' + toString(c.catid) AS recommended_context,
     COLLECT(pt.name + ': ' + pv.value) AS viewed_item_properties
     
// Final Prompt Construction
RETURN 'The user was viewing Product 119736 with the following recent properties: ' + 
        apoc.text.join(viewed_item_properties, ', ') + 
        '. Graph reasoning based on co-occurrence suggests recommending Product 186702 which is the ' + recommended_context + 
        '. The average time between these views is ~2.62 minutes. Explain why this recommendation is highly probable and logical, focusing on the shared context.' AS LLM_Prompt
"""

# Re-initialize driver
try:
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    # Run the query
    prompt_result = run_cypher_query(hybrid_query)
    
    if prompt_result:
        print("Generated LLM Prompt (The Input for your LLM):")
        print("------------------------------------------------------------------------------------------------------------------------------------------------------------------")
        print(prompt_result[0]['LLM_Prompt'])
    else:
        print("Could not retrieve context data for the prompt.")
        
except Exception as e:
    print(f"ERROR running hybrid query: {e}")
    # This might fail if the apoc.text.join is not installed. If so, a simpler string concatenation is needed.

finally:
    if 'driver' in locals() and driver:
        driver.close()


--- 6. Generating LLM Prompt for Recommendation Explanation ---
Generated LLM Prompt (The Input for your LLM):
------------------------------------------------------------------------------------------------------------------------------------------------------------------
The user was viewing Product 119736 with the following recent properties: 888: n4080.000, 790: n4080.000, 917: n4080.000, 202: n4080.000, 909: n4080.000. Graph reasoning based on co-occurrence suggests recommending Product 186702 which is the Recommended Product (186702) in Category 57. The average time between these views is ~2.62 minutes. Explain why this recommendation is highly probable and logical, focusing on the shared context.


## Setup and Test Set Selection

In [14]:
import numpy as np
from neo4j import GraphDatabase, exceptions
from tqdm import tqdm

# --- Re-initialize Driver (Make sure this is at the start of a new cell) ---
URI = "bolt://127.0.0.1:7687"
AUTH = ("neo4j", "12345678")
driver = GraphDatabase.driver(URI, auth=AUTH)

def run_cypher_query(query, params=None, timeout_s=3600):
    with driver.session() as session:
        result = session.run(query, params, timeout=timeout_s)
        return [record for record in result]

# --- 1. Define Test Set (10% of sessions) ---
# Use the session_nodes_df we created earlier (1,930,778 total sessions)
unique_sids = session_nodes_df['sid'].unique()
TEST_SET_SIZE = int(len(unique_sids) * 0.10)

# Shuffle and select the test sessions
np.random.seed(42) # For reproducibility
test_sids = np.random.choice(unique_sids, size=TEST_SET_SIZE, replace=False)

# Filter the main events DataFrame for the test sessions
test_events_df = events_df[events_df['sid'].isin(test_sids)].copy()

print(f"Total sessions in Test Set: {TEST_SET_SIZE:,}")
print(f"Total events in Test Set: {len(test_events_df):,}")

Total sessions in Test Set: 193,077
Total events in Test Set: 275,594


## MRR Simulation and Calculation

In [15]:
# Function to get the predicted next items for a given current product ID
def get_prediction_ranking(current_pid, k=50): # Set a high K to check for the correct item
    """Queries the graph for the top K co-occurring products by popularity."""
    query = f"""
    MATCH (p1:Product {{pid: $pid}})-[co:CO_OCCURS_WITH]-(p2:Product)
    RETURN p2.pid AS recommended_pid
    ORDER BY co.popularity DESC
    LIMIT $k
    """
    try:
        results = run_cypher_query(query, {'pid': current_pid, 'k': k}, timeout_s=120) # Use a shorter timeout for this frequent query
        return [rec['recommended_pid'] for rec in results]
    except Exception as e:
        # print(f"Prediction Query Error for PID {current_pid}: {e}")
        return []

# --- 2. MRR Calculation Logic ---

test_sessions = test_events_df.groupby('sid')
total_rr = 0
total_predictions = 0

print("\n--- Starting MRR Prediction Simulation ---")
# Use tqdm to track progress over the test sessions
for sid, session_data in tqdm(test_sessions, desc="Calculating MRR"):
    
    # Sort events by step to ensure correct sequence
    sorted_session = session_data.sort_values('step').reset_index(drop=True)
    
    # Iterate through all events except the last one.
    # We predict the item at 'i+1' based on the item at 'i'.
    # The paper excludes the first event (index 0) as it cannot be predicted.
    for i in range(len(sorted_session) - 1):
        
        # 1. Prediction Context (Current Item)
        current_pid = sorted_session.loc[i, 'pid']
        
        # 2. Ground Truth (Actual Next Item)
        actual_next_pid = sorted_session.loc[i+1, 'pid']
        
        # 3. Get Prediction from the Graph
        predicted_rank_list = get_prediction_ranking(current_pid, k=50) # Predict top 50
        
        # 4. Find the rank of the actual next item
        total_predictions += 1
        
        try:
            # Python list index starts at 0, rank starts at 1
            rank = predicted_rank_list.index(actual_next_pid) + 1 
            
            # 5. Calculate Reciprocal Rank and add to total
            reciprocal_rank = 1 / rank
            total_rr += reciprocal_rank
            
        except ValueError:
            # Actual next item was not found in the top 50, so RR is 0
            continue 

# Final MRR Calculation
if total_predictions > 0:
    final_mrr = total_rr / total_predictions
else:
    final_mrr = 0

driver.close()
print("\n--- MRR Calculation Complete ---")
print(f"Total Predictions Made: {total_predictions:,}")
print(f"Final Mean Reciprocal Rank (MRR): {final_mrr:.4f}")


--- Starting MRR Prediction Simulation ---


Calculating MRR: 100%|██████████| 193077/193077 [07:41<00:00, 418.37it/s]


--- MRR Calculation Complete ---
Total Predictions Made: 82,517
Final Mean Reciprocal Rank (MRR): 0.2503
